# 6. Attempts, effort and lessons learned

This page documents historical Colab experiments and the reasons for changing the pipeline. These are smoke/partition observations, not the final 20,000-ID benchmark or controlled A/B tests. Revisions, selected IDs, network conditions and measurement boundaries differ. Development stages are not estimates of person-hours or individual contributions.

The CSV below is extracted from saved benchmark bundles, with run IDs, code revisions and SHA-256 provenance in `data/m1/effort/`. The local raw bundles are intentionally not committed. Tables and charts are generated when the Book executes this notebook.

Drafted with assistance from OpenAI Codex, disclosed per course rules.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "myst.yml").exists())
attempts = pd.read_csv(root / "data/m1/effort/attempts.csv")
table = attempts[["experiment", "required", "complete", "wall_s", "scope", "discovery_s", "retry_backoff_s"]].copy()
for col in ("wall_s", "discovery_s", "retry_backoff_s"):
    table[col] = pd.to_numeric(table[col], errors="coerce").round().astype("Int64")
display(HTML('<div class="table-scroll">' + table.to_html(index=False, na_rep="N/A") + "</div>"))

display(HTML("<p class='table-caption'>Table 6.1. Historical experiments, coverage and recorded timings; scope differs across runs.</p>"))


## 6.1. API discovery and source throttling

The earliest saved successful API/OAI + Drive evidence covers 200 IDs, with two chunk profiles of 895.38 and 909.80 seconds (1,805.18 seconds total), including 318.8 seconds of logged backoff. This is a legacy chunk-only boundary. No saved evidence establishes a pure API baseline of approximately 650 seconds per 100 IDs.

A separate 100-ID attempt spent 1,531.11 observed session seconds without issuing a source request. Its rescue analysis records 11 read timeouts; the discovery log includes API 429 and OAI 503 responses. This documents a discovery failure, not successful processing throughput.

Snapshot discovery was introduced before the successful HF experiment. The 100-ID snapshot + Drive run completed in 635.36 session seconds; the snapshot + HF run completed in 502.10 session seconds. These observations do not isolate the effect of the output backend.


## 6.2. Hugging Face experiments

HF was evaluated to reduce repeated small-file synchronization through Drive FUSE. The successful 100-ID HF run had complete timing evidence. A later filtered-input attempt reached a verified post-chunk checkpoint but the operator supplied a traceback showing HTTP 429 on final report publication: the service reported a quota of **128 repository commit creations per hour**. That historical error concerns commits, not all HTTP requests; its traceback has not been located in the archived bundles used for this page.

Increasing the next experiment to 500 selected IDs did not demonstrate a discovery failure: its console reports `500/500 IDs found in the Kaggle snapshot` and then source downloads. Source `/src/` requests encountered 429 and transfer errors. The saved bundle has no completed performance summary, so it is excluded from the performance chart. A larger batch cannot be identified as the cause of discovery failure from this evidence.


## 6.3. Offline metadata and filtered inputs

Valid snapshot records reduce online discovery requests, but historical titles/authors and missing/invalid records can still require API/OAI calls and encounter throttling. We do not claim to eliminate every discovery 429.

The full input occupied several GB. The filtered input retains complete records for all 20,000 assigned IDs: 39,990,663 bytes and 26,491 versions as of that snapshot. The parser streams JSONL; filtering reduces disk and scanning work, rather than replacing a mandatory full-file RAM load. The snapshot cannot prove the absence of later revisions.

The full-input HF smoke measured 5,708.68 MB sampled peak disk. The earlier Drive snapshot smoke reported 20.50 MB under an older boundary which omitted the full input. Those disk values must not be interpreted as a storage-backend comparison.


## 6.4. Returning to Drive: healthy early chunks did not predict completion

The run `m1-dang-1000-01` requested 1,000 IDs in 100-ID chunks. It completed 399 of the 400 IDs reached before stopping, after 3,252.16 observed session seconds (about 54 minutes). Chunk profile times were 404.56, 476.85, 937.07 and 1,415.56 seconds: about 6.74, 7.95, 15.62 and 23.59 minutes.

The missing source was `2604.20926v3` (paper offset 391), after five HTTP 429 attempts. The primary error was incomplete chunk QA; cleanup recorded no error. Thus this artifact does not establish a final sync failure. Its timing audit is incomplete, so the session figure is an observation, not a complete benchmark of 1,000 IDs.

A subsequent snapshot + Drive experiment on offsets 500–999 completed 500/500 IDs in 3,954.65 session seconds with two profiler attempts. The retry/backoff summary recorded zero seconds. This successful experiment is retained in the history alongside failures.


In [ ]:
chunk_profiles = pd.read_csv(root / "data/m1/effort/dang_1000_chunk_profiles.csv")
labels = chunk_profiles.offset_start.astype(str) + "–" + chunk_profiles.offset_end.astype(str)
chunk_minutes = pd.Series(chunk_profiles.wall_s.to_numpy() / 60, index=labels)
fig, ax = plt.subplots(figsize=(7, 3.3))
ax.bar(chunk_minutes.index, chunk_minutes.values, color=["#357a9e"] * 3 + ["#bd6b35"])
ax.set(xlabel="Attempted offsets within Dang's range", ylabel="Chunk profiler wall (minutes)", title="Incomplete 1,000-ID attempt: growing chunk wall time")
for i, value in enumerate(chunk_minutes):
    ax.text(i, value + .3, f"{value:.2f}", ha="center")
ax.set_ylim(0, 27)
fig.tight_layout()
plt.show()


## 6.5. Hybrid bulk S3 and version-specific arXiv downloads

For papers with only v1 in the snapshot, the pipeline permits a timestamp/manifest-based inference and records `inferred_snapshot_bundle_timestamp`; the bulk member does not necessarily supply an explicit version label. The inference is conditional on a complete snapshot reflecting at least the bundle cutoff. The exact snapshot cutoff relative to each bundle has not been established; matching acquisition dates do not prove it. The conditional argument and the implemented heuristic are detailed in [Methodology](01-methodology.md).

During S3 preparation, the first checked bundle, `arXiv_src_2604_148.tar`, exposed a discrepancy: the manifest listed 530,006,104 bytes, whereas the Drive file was 530,104,320 bytes (98,216 bytes more). The operator's verification output reported that the actual file MD5 matched the manifest MD5, `4c14aeea5aee5105e87b828440595456`. The saved hybrid S3 index independently records that digest and the Drive file size. We retained both sizes and used the matching checksum in the bundle-validation policy; the size difference alone did not establish a corrupt download. This observation does not establish why the sizes differ or prove the validity of other bundles.

For multi-version papers, the pipeline requests each version explicitly from arXiv. A monthly bulk bundle may retain a newer submission in place of an earlier same-month version and cannot be assumed to contain the complete version history. The hybrid ordering processes the selected only-v1 group first, followed by fallback and multi-version work.

The first 1,000-ID hybrid sample (offsets 0–999) completed in 4,453.87 session seconds (1h 14m 13.87s), with 700 inferred S3 v1 records and 470.1 seconds of logged backoff. The next partition (offsets 1000–1999) completed in 3,517.46 seconds (58m 37.46s), with 704 inferred S3 v1 records and zero logged backoff. The approximately 21% lower session duration is an observation across different ID sets and code revisions, not a causal speedup estimate.

## 6.6. From hybrid experiments to the official split runs

The group subsequently organized collection as one only-v1 run using S3 and arXiv fallback, plus four member-specific multi-version runs using version-specific arXiv requests. This split describes the final run organization; its full coverage, QA and performance are reported separately in [Data statistics](02-data-stats.ipynb) and [Performance](03-performance.ipynb) once the official evidence has been reconciled. Historical samples below are not substitutes for those results.


In [ ]:
complete_sessions = attempts.loc[(attempts["scope"] == "session wall") & attempts["coverage_complete"]].copy()
complete_sessions["seconds_per_required_id"] = complete_sessions["wall_s"] / complete_sessions["required"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(complete_sessions["experiment"], complete_sessions["seconds_per_required_id"], color="#357a9e")
ax.set(xlabel="Observed session seconds / required ID", title="Historical completed sessions (different inputs and revisions)")
ax.invert_yaxis()
fig.tight_layout()
plt.show()


## 6.7. Evidence and interpretation

The next table identifies each archived run and implementation revision. `complete` means the pipeline's terminal source/QA definition; it includes explicit `no_tex` and `unavailable` outcomes and does not prove complete reference metadata. Read coverage, QA and timing audits together. The hybrid samples do not establish full-corpus coverage or final performance; they also exclude environment setup and previously prepared bulk downloads.

Future checkpoint/shard and fast-resume changes remain planned work. They are not credited as measured improvements on this page. Resume time, restoration and synchronization overhead must remain visible in future benchmarks.


In [ ]:
display(HTML('<div class="table-scroll">' + attempts[["evidence", "run_id", "revision", "offset", "required", "complete", "timing_complete", "s3_inferred"]].to_html(index=False, na_rep="N/A") + "</div>"))

display(HTML("<p class='table-caption'>Table 6.2. Evidence identifiers, revisions and coverage of historical experiments.</p>"))
